# 1단계 분류기 학습 — klue/roberta-base (Colab)

학부모 민원을 **정상/주의/위험** 3클래스로 분류. AI Hub 감성대화 말뭉치로 파인튜닝.

**실행 순서**: 위에서부터 셀 차례로 실행 (런타임 → 모두 실행).
- 런타임 유형: **GPU**(T4면 충분) 로 변경 후 시작.
- 라벨 순서 `0=정상,1=주의,2=위험` 고정 → 백엔드에 그대로 연동.


## 1) 라이브러리 설치


In [ ]:
!pip -q install "transformers>=4.44" "datasets>=2.20" scikit-learn accelerate

## 2) 데이터 업로드
AI Hub 감성대화 **라벨링데이터** JSON 2개(Training/Validation)를 올린다.
- 방법 A: 아래 셀 실행 → 파일 선택(`...Training.json`, `...Validation.json`)
- 방법 B(권장, 파일 큼): 구글 드라이브에 올린 뒤 `drive.mount` 후 경로 지정


In [ ]:
# 방법 A: 직접 업로드 (Training 41MB라 느릴 수 있음)
from google.colab import files
up = files.upload()
paths = list(up.keys())
TRAIN_JSON = [p for p in paths if 'Train' in p or 'train' in p][0]
VAL_JSON   = [p for p in paths if 'Valid' in p or 'val' in p][0]
print('train:', TRAIN_JSON, '| val:', VAL_JSON)

In [ ]:
# 방법 B(주석 해제): 드라이브 사용
# from google.colab import drive; drive.mount('/content/drive')
# TRAIN_JSON = '/content/drive/MyDrive/aihub/감성대화말뭉치(최종데이터)_Training.json'
# VAL_JSON   = '/content/drive/MyDrive/aihub/감성대화말뭉치(최종데이터)_Validation.json'

## 3) 라벨 매핑 + JSON → (text, label)
감정코드 6대분류(십의 자리) → 3클래스: 분노→위험 / 슬픔·불안·상처·당황→주의 / 기쁨→정상


In [ ]:
import json
from collections import Counter

LABELS = ['정상','주의','위험']
LABEL2ID = {l:i for i,l in enumerate(LABELS)}   # 0정상 1주의 2위험 (백엔드 규약)
ID2LABEL = {i:l for l,i in LABEL2ID.items()}

CATEGORY_BY_DECADE = {1:'분노',2:'슬픔',3:'불안',4:'상처',5:'당황',6:'기쁨'}
CAT2CLASS = {'분노':'위험','슬픔':'주의','불안':'주의','상처':'주의','당황':'주의','기쁨':'정상'}

def emo_to_label(code):
    d = ''.join(c for c in str(code) if c.isdigit())
    if not d: return None
    cat = CATEGORY_BY_DECADE.get(int(d)//10)
    return CAT2CLASS.get(cat) if cat else None

def load_rows(path):
    data = json.load(open(path, encoding='utf-8'))
    items = data if isinstance(data, list) else data.get('data', [])
    rows = []
    for it in items:
        content = (it.get('talk') or {}).get('content') or {}
        text = ' '.join(v.strip() for k,v in content.items()
                        if isinstance(v,str) and v.strip() and k.upper().startswith('HS'))
        emo = ((it.get('profile') or {}).get('emotion') or {}).get('type','')
        lab = emo_to_label(emo)
        if text and lab in LABEL2ID:
            rows.append({'text':text, 'labels':LABEL2ID[lab]})
    return rows

train_rows = load_rows(TRAIN_JSON)
val_rows   = load_rows(VAL_JSON)
print('train', len(train_rows), Counter(ID2LABEL[r['labels']] for r in train_rows))
print('val  ', len(val_rows),   Counter(ID2LABEL[r['labels']] for r in val_rows))

## 4) 토크나이즈 (klue/roberta-base)


In [ ]:
from datasets import Dataset
from transformers import AutoTokenizer

MODEL = 'klue/roberta-base'
tok = AutoTokenizer.from_pretrained(MODEL)

def to_ds(rows):
    ds = Dataset.from_list(rows)
    return ds.map(lambda b: tok(b['text'], truncation=True, max_length=128), batched=True)

train_ds = to_ds(train_rows)
val_ds   = to_ds(val_rows)

## 5) 클래스 가중치 (불균형 보정)
주의 70% 쏠림 → 역빈도 가중치 + 위험 부스트(위험→정상 오분류 최소화)


In [ ]:
import numpy as np, torch
from collections import Counter
cnt = Counter(r['labels'] for r in train_rows)
N = sum(cnt.values()); DANGER_BOOST = 1.3
weights = []
for i in range(3):
    w = N/(3*cnt[i]) if cnt[i] else 1.0
    if i==2: w*=DANGER_BOOST
    weights.append(w)
print({ID2LABEL[i]:round(w,3) for i,w in enumerate(weights)})
class_weights = torch.tensor(weights, dtype=torch.float)

## 6) 학습 (가중 손실 + Macro-F1 평가)


In [ ]:
from transformers import (AutoModelForSequenceClassification, TrainingArguments,
                          Trainer, DataCollatorWithPadding)
from sklearn.metrics import f1_score, recall_score

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL, num_labels=3, id2label=ID2LABEL, label2id=LABEL2ID)

class WeightedTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kw):
        labels = inputs.pop('labels')
        out = model(**inputs)
        loss = torch.nn.functional.cross_entropy(
            out.logits, labels, weight=class_weights.to(out.logits.device))
        return (loss, out) if return_outputs else loss

def metrics(p):
    pred = p.predictions.argmax(-1)
    return {'macro_f1': f1_score(p.label_ids, pred, average='macro'),
            'danger_recall': recall_score(p.label_ids, pred, labels=[2], average='macro', zero_division=0)}

args = TrainingArguments(
    output_dir='out', num_train_epochs=3, per_device_train_batch_size=32,
    per_device_eval_batch_size=64, learning_rate=2e-5, weight_decay=0.01,
    eval_strategy='epoch', save_strategy='epoch', load_best_model_at_end=True,
    metric_for_best_model='macro_f1', logging_steps=50, fp16=True, report_to=[])

trainer = WeightedTrainer(model=model, args=args, train_dataset=train_ds,
    eval_dataset=val_ds, compute_metrics=metrics,
    data_collator=DataCollatorWithPadding(tok))
trainer.train()

## 7) 평가 — 혼동행렬 + 위험→정상 오분류(최악 FN)


In [ ]:
from sklearn.metrics import confusion_matrix, classification_report
pred = trainer.predict(val_ds)
y_pred = pred.predictions.argmax(-1); y_true = pred.label_ids
cm = confusion_matrix(y_true, y_pred, labels=[0,1,2])
print('혼동행렬 (행=실제, 열=예측) [정상,주의,위험]')
print(cm)
print('\n🚨 위험→정상 오분류:', int(cm[2][0]), '건')
print()
print(classification_report(y_true, y_pred, target_names=LABELS, digits=3))

## 8) 저장 → 다운로드 (백엔드 연동용)
모델+토크나이저를 한 폴더에 저장하고 zip으로 받는다.
백엔드: `CLASSIFIER_MODEL_DIR=/압축푼/경로 uvicorn app.main:app --reload`


In [ ]:
SAVE='klue-roberta-clf'
trainer.save_model(SAVE)
tok.save_pretrained(SAVE)
!zip -qr klue-roberta-clf.zip {SAVE}
from google.colab import files
files.download('klue-roberta-clf.zip')
print('완료 — 압축 풀어서 그 폴더 경로를 CLASSIFIER_MODEL_DIR 로 지정')